# 00_setup - Aprovisionamiento inicial FinPay Lakehouse

Este notebook se ejecuta **una sola vez** antes de desplegar o ejecutar el Databricks Asset Bundle.

Objetivo:
- Crear catálogo, schemas y volume de landing.
- Crear subdirectorios del volume.
- Configurar permisos por rol.
- Crear funciones de seguridad para PII.
- Preparar una tabla `silver.users` con column masking y row-level security.
- Crear archivo metadata-driven `ingestion_archetypes.json`.
- Validar el entorno base.

## 1. Parámetros del entorno

In [ ]:
CATALOG = "fintech_finpay"

SCHEMA_DEFAULT = "default"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
SCHEMA_OBSERVABILITY = "observability"

VOLUME_NAME = "vol_landing"
LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA_DEFAULT}/{VOLUME_NAME}"

# Roles/grupos sugeridos para el proyecto.
# Deben existir como Account Groups en Databricks.
GROUP_INGENIERIA = "finpay_ingenieria"
GROUP_RIESGO = "finpay_riesgo"
GROUP_AUDITORIA = "finpay_auditoria"

print(f"Catálogo      : {CATALOG}")
print(f"Landing path  : {LANDING_PATH}")
print(f"Grupo ingeniería: {GROUP_INGENIERIA}")
print(f"Grupo riesgo     : {GROUP_RIESGO}")
print(f"Grupo auditoría  : {GROUP_AUDITORIA}")


## 2. Crear catálogo, schemas y volume

El schema `default` se crea explícitamente porque el volume `vol_landing` vivirá dentro de `fintech_finpay.default`.


In [ ]:

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

for schema_name in [
    SCHEMA_DEFAULT,
    SCHEMA_BRONZE,
    SCHEMA_SILVER,
    SCHEMA_GOLD,
    SCHEMA_OBSERVABILITY
]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}")

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME}
COMMENT 'Landing zone para archivos fuente del proyecto FinPay'
""")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA_DEFAULT}"))


## 3. Crear subdirectorios del volume

Estructura recomendada:

```text
/Volumes/fintech_finpay/default/vol_landing/
├── transactions/
├── merchants/
├── users/
├── metadata/
├── _schemas/
├── _checkpoints/
└── quarantine/
```

- `transactions/`, `merchants/`, `users/`: archivos fuente.
- `metadata/`: configuración de arquetipos de ingesta.
- `_schemas/`: ubicaciones de inferencia/evolución de esquemas de Auto Loader.
- `_checkpoints/`: checkpoints de streaming.
- `quarantine/`: archivos o trazas auxiliares de registros rechazados.


In [ ]:
# COMMAND ----------
# DBTITLE 1,Crear carpetas de landing
directories = [
    f"{LANDING_PATH}/transactions",
    f"{LANDING_PATH}/merchants",
    f"{LANDING_PATH}/users",
    f"{LANDING_PATH}/metadata",
    f"{LANDING_PATH}/_schemas",
    f"{LANDING_PATH}/_schemas/transactions",
    f"{LANDING_PATH}/_schemas/merchants",
    f"{LANDING_PATH}/_schemas/users",
    f"{LANDING_PATH}/_checkpoints",
    f"{LANDING_PATH}/_checkpoints/bronze_transactions",
    f"{LANDING_PATH}/_checkpoints/bronze_merchants",
    f"{LANDING_PATH}/_checkpoints/bronze_users",
    f"{LANDING_PATH}/quarantine",
]

for path in directories:
    dbutils.fs.mkdirs(path)

display(dbutils.fs.ls(LANDING_PATH))


## 4. Crear archivo `ingestion_archetypes.json`

Este archivo permite que el pipeline sea **metadata-driven**: si se agrega una nueva fuente, se actualiza el JSON sin modificar el código principal del pipeline.


In [ ]:
# COMMAND ----------
# DBTITLE 1,Crear metadata de arquetipos de ingesta
import json

archetypes = [
    {
        "source_name": "transactions",
        "source_path": f"{LANDING_PATH}/transactions/",
        "file_format": "csv",
        "delimiter": ",",
        "header": True,
        "target_table": "bronze_transactions",
        "schema_location": f"{LANDING_PATH}/_schemas/transactions",
        "checkpoint_path": f"{LANDING_PATH}/_checkpoints/bronze_transactions",
        "active": True
    },
    {
        "source_name": "merchants",
        "source_path": f"{LANDING_PATH}/merchants/",
        "file_format": "json",
        "delimiter": None,
        "header": None,
        "target_table": "bronze_merchants",
        "schema_location": f"{LANDING_PATH}/_schemas/merchants",
        "checkpoint_path": f"{LANDING_PATH}/_checkpoints/bronze_merchants",
        "active": True
    },
    {
        "source_name": "users",
        "source_path": f"{LANDING_PATH}/users/",
        "file_format": "csv",
        "delimiter": "|",
        "header": True,
        "target_table": "bronze_users",
        "schema_location": f"{LANDING_PATH}/_schemas/users",
        "checkpoint_path": f"{LANDING_PATH}/_checkpoints/bronze_users",
        "active": True
    }
]

metadata_path = f"{LANDING_PATH}/metadata/ingestion_archetypes.json"

dbutils.fs.put(
    metadata_path,
    json.dumps(archetypes, indent=2, ensure_ascii=False),
    overwrite=True
)

print(f"Archivo creado: {metadata_path}")
print(dbutils.fs.head(metadata_path, 4000))


## 5. Permisos por rol

Matriz solicitada:

| Rol | Permisos mínimos |
|---|---|
| Ingeniería | `USE CATALOG`, `USE SCHEMA`, `CREATE TABLE`, `MODIFY` en todos los schemas |
| Riesgo | `USE CATALOG`, `USE SCHEMA`, `SELECT` en `silver` y `gold` |
| Auditoría | `USE CATALOG`, `USE SCHEMA`, `SELECT` en `gold` y `observability` |

También se otorgan permisos sobre el volume para que ingeniería pueda cargar y administrar archivos.


In [ ]:
# COMMAND ----------
# DBTITLE 1,Aplicar grants sobre catálogo y schemas
# Nota:
# - Los grupos deben existir previamente.
# - Ejecutar este bloque con un usuario owner/admin del catálogo o con permisos suficientes.

def grant(sql_statement: str):
    print(sql_statement)
    spark.sql(sql_statement)

# Uso del catálogo
for group_name in [GROUP_INGENIERIA, GROUP_RIESGO, GROUP_AUDITORIA]:
    grant(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{group_name}`")

# Ingeniería: acceso operativo completo a todos los schemas del proyecto
for schema_name in [SCHEMA_DEFAULT, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_OBSERVABILITY]:
    grant(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT CREATE TABLE ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT CREATE FUNCTION ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT MODIFY ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT SELECT ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")

# Riesgo: lectura en silver y gold
for schema_name in [SCHEMA_SILVER, SCHEMA_GOLD]:
    grant(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_RIESGO}`")
    grant(f"GRANT SELECT ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_RIESGO}`")

# Auditoría: lectura en gold y observability
for schema_name in [SCHEMA_GOLD, SCHEMA_OBSERVABILITY]:
    grant(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_AUDITORIA}`")
    grant(f"GRANT SELECT ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_AUDITORIA}`")

# Permisos sobre el volume de landing
grant(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_INGENIERIA}`")
grant(f"GRANT WRITE VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_INGENIERIA}`")

# Riesgo y auditoría solo lectura del landing si el curso/proyecto exige trazabilidad.
# Si no deseas que vean archivos crudos, comenta estas dos líneas.
grant(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_RIESGO}`")
grant(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_AUDITORIA}`")


## 6. Funciones de seguridad para column masking y row-level security

Se crearán funciones SQL en `fintech_finpay.silver`.

Regla:
- `finpay_ingenieria` ve los valores reales de campos PII.
- Otros roles ven valores enmascarados.
- Para RLS, ingeniería ve todos los países. Riesgo y auditoría ven filas de países permitidos mediante una función simple.

> En un caso real, la regla de RLS se puede reemplazar por una tabla de asignación `user -> country` o `group -> country`.


In [ ]:
# COMMAND ----------
# DBTITLE 1,Crear funciones de masking y RLS
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA_SILVER}.mask_pii_string(value STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('{GROUP_INGENIERIA}') THEN value
    WHEN value IS NULL THEN NULL
    ELSE '***MASKED***'
  END
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA_SILVER}.mask_email(value STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('{GROUP_INGENIERIA}') THEN value
    WHEN value IS NULL THEN NULL
    WHEN instr(value, '@') > 1 THEN concat('***@', split(value, '@')[1])
    ELSE '***MASKED***'
  END
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA_SILVER}.mask_phone(value STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('{GROUP_INGENIERIA}') THEN value
    WHEN value IS NULL THEN NULL
    WHEN length(value) >= 4 THEN concat('***', right(value, 4))
    ELSE '***MASKED***'
  END
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA_SILVER}.users_country_filter(country STRING)
RETURNS BOOLEAN
RETURN
  CASE
    WHEN is_account_group_member('{GROUP_INGENIERIA}') THEN TRUE
    WHEN is_account_group_member('{GROUP_RIESGO}') THEN country IN ('PE', 'CO', 'MX', 'CL', 'AR')
    WHEN is_account_group_member('{GROUP_AUDITORIA}') THEN country IN ('PE', 'CO', 'MX', 'CL', 'AR')
    ELSE FALSE
  END
""")

display(spark.sql(f"SHOW FUNCTIONS IN {CATALOG}.{SCHEMA_SILVER}"))


## 7. Crear tabla base `silver.users` con PII protegido

Para dejar la seguridad preparada desde el setup, creamos la tabla con columnas esperadas, máscaras y row filter.

Campos PII:
- `full_name`
- `document_id`
- `email`
- `phone`


In [ ]:
# COMMAND ----------
# DBTITLE 1,Crear tabla silver.users con column masks y row filter
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}.users (
  user_id STRING COMMENT 'Identificador único del usuario',
  full_name STRING MASK {CATALOG}.{SCHEMA_SILVER}.mask_pii_string COMMENT 'Nombre completo del usuario - PII',
  document_id STRING MASK {CATALOG}.{SCHEMA_SILVER}.mask_pii_string COMMENT 'Documento de identidad - PII',
  email STRING MASK {CATALOG}.{SCHEMA_SILVER}.mask_email COMMENT 'Correo electrónico - PII',
  phone STRING MASK {CATALOG}.{SCHEMA_SILVER}.mask_phone COMMENT 'Teléfono - PII',
  country STRING COMMENT 'País de residencia',
  segment STRING COMMENT 'Segmento comercial',
  registration_date DATE COMMENT 'Fecha de registro',
  _ingested_at TIMESTAMP COMMENT 'Fecha/hora de ingesta',
  _source_file STRING COMMENT 'Archivo origen'
)
USING DELTA
COMMENT 'Usuarios FinPay limpios con protección de PII mediante column masking y row-level security'
TBLPROPERTIES (
  'quality' = 'silver',
  'delta.enableChangeDataFeed' = 'true'
)
""")

spark.sql(f"""
ALTER TABLE {CATALOG}.{SCHEMA_SILVER}.users
SET ROW FILTER {CATALOG}.{SCHEMA_SILVER}.users_country_filter ON (country)
""")

display(spark.sql(f"DESCRIBE EXTENDED {CATALOG}.{SCHEMA_SILVER}.users"))


## 8. Crear tabla de cuarentena en Silver

Esta tabla almacenará registros rechazados por reglas críticas de calidad.


In [ ]:
# COMMAND ----------
# DBTITLE 1,Crear tabla de cuarentena
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}.quarantine (
  source_name STRING COMMENT 'Fuente original del registro',
  target_table STRING COMMENT 'Tabla destino esperada',
  rejection_reason STRING COMMENT 'Motivo de rechazo',
  processed_at TIMESTAMP COMMENT 'Fecha/hora de procesamiento',
  raw_record STRING COMMENT 'Contenido original del registro en formato STRING',
  source_file STRING COMMENT 'Ruta del archivo origen'
)
USING DELTA
COMMENT 'Tabla de cuarentena para registros rechazados por reglas de calidad en Silver'
TBLPROPERTIES (
  'quality' = 'silver',
  'delta.enableChangeDataFeed' = 'true'
)
""")

display(spark.sql(f"DESCRIBE EXTENDED {CATALOG}.{SCHEMA_SILVER}.quarantine"))


## 9. Crear tabla de observabilidad base

El pipeline debe persistir event logs en `fintech_finpay.observability`.  
Esta tabla base permite validar permisos y tener un punto estándar para métricas auxiliares del proyecto.


In [ ]:
# COMMAND ----------
# DBTITLE 1,Crear tabla base de observabilidad
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA_OBSERVABILITY}.pipeline_run_audit (
  run_id STRING,
  pipeline_name STRING,
  layer STRING,
  dataset_name STRING,
  status STRING,
  records_processed BIGINT,
  records_failed BIGINT,
  started_at TIMESTAMP,
  ended_at TIMESTAMP,
  created_at TIMESTAMP
)
USING DELTA
COMMENT 'Tabla auxiliar de auditoría de ejecuciones del pipeline FinPay'
TBLPROPERTIES (
  'quality' = 'observability'
)
""")

display(spark.sql(f"DESCRIBE EXTENDED {CATALOG}.{SCHEMA_OBSERVABILITY}.pipeline_run_audit"))


## 10. Validaciones finales

Este bloque verifica:
- Catálogo y schemas.
- Volume y carpetas.
- Archivo de arquetipos.
- Grants principales.
- Funciones de seguridad.
- Tablas base.


In [ ]:
# COMMAND ----------
# DBTITLE 1,Validación final del aprovisionamiento
print("=== Schemas ===")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

print("=== Volumes ===")
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA_DEFAULT}"))

print("=== Landing folders ===")
display(dbutils.fs.ls(LANDING_PATH))

print("=== Metadata file ===")
print(dbutils.fs.head(f"{LANDING_PATH}/metadata/ingestion_archetypes.json", 4000))

print("=== Tables Silver ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_SILVER}"))

print("=== Tables Observability ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_OBSERVABILITY}"))

print("=== Grants Catalog ===")
display(spark.sql(f"SHOW GRANTS ON CATALOG {CATALOG}"))

print("=== Grants Silver ===")
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG}.{SCHEMA_SILVER}"))

print("Aprovisionamiento base finalizado correctamente.")


## 11. Siguiente paso

Después de ejecutar correctamente este notebook:

1. Sube los archivos fuente a:
   - `/Volumes/fintech_finpay/default/vol_landing/transactions/`
   - `/Volumes/fintech_finpay/default/vol_landing/merchants/`
   - `/Volumes/fintech_finpay/default/vol_landing/users/`

2. Ejecuta:
   ```bash
   databricks bundle validate --profile finpay-lakehouse
   ```

3. Luego despliega el DAB:
   ```bash
   databricks bundle deploy --target dev --profile finpay-lakehouse
   ```
